# Baseline GPT-2 cho Vietnamese Math Word Problems — V2 (Upgraded)

Notebook V2 fine-tune `NlpHUST/gpt2-vietnamese` để sinh **lời giải toán tiếng Việt + đáp án cuối**:

```text
Lời giải ...
Đáp án là: <số>
```

**Nâng cấp so với baseline:**

1. **Prompt mới**: có instruction + type hint tiếng Việt → model học format chắc hơn.
2. **Training**: 2 epochs, LR=5e-5, effective batch 32, bf16, `adamw_torch_fused`.
3. **Decoding**: beam search (num_beams=4) + batch inference + custom StoppingCriteria. Bỏ `no_repeat_ngram_size`, bỏ `repetition_penalty`. `max_new_tokens=320`.
4. **Eval fix**: cho phép `allow_last_number=True` cho prediction; auto-append anchor nếu output chỉ có số.
5. **EOS robust**: dùng `tokenizer.eos_token_id` thật, resize model embedding theo `len(tokenizer)` để tránh CUDA index out of bounds.


## Run

Kaggle: GPU ON, Internet OFF. Tổng thời gian dự kiến **<= 3 giờ**.

Output:
- `data/train_preprocessed.json`: train set sau preprocessing và smart truncation.
- `data/train_preprocessing_report.json`: thống kê preprocessing.
- `gpt2_math_baseline_ckpt/`: checkpoint sau fine-tune.
- `valid_output.json` + `valid_report.json`: output và đánh giá chi tiết validation.
- `test_predictions.json`: file nộp cho Phase 2.


In [ ]:
# 1. Import và kiểm tra môi trường
import os
import sys
import gc
import re
import json
import math
import time
import random
import hashlib
import inspect
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)

try:
    from IPython.display import display
except Exception:
    display = print

try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

pd.set_option("display.max_colwidth", 180)

print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
CUDA_OK = torch.cuda.is_available()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        capability = torch.cuda.get_device_capability(i)
        print(f"GPU {i}:", name, "| capability:", capability)
    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        CUDA_OK = False
        print("WARNING: GPU hiện tại có compute capability < 7.0, không tương thích với PyTorch CUDA hiện tại.")
        print("Hãy chọn GPU T4/V100/A100 thay vì P100 trên Kaggle.")


In [ ]:
# 2. Đường dẫn dữ liệu, model và output
def first_existing(*paths):
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào:\n" + "\n".join(checked))


def first_existing_optional(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
)

MODEL_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese",
    PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
)

WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "baseline_gpt2_math_v2"
WORK_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_DATA_DIR = WORK_DIR / "data" if IS_KAGGLE else PROJECT_ROOT / "data"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"
TEST_FILE = first_existing_optional(DATA_DIR / "test.json", "/kaggle/input/test.json")

OUTPUT_DIR = WORK_DIR / "gpt2_math_baseline_ckpt"
VALID_OUTPUT_PATH = WORK_DIR / "valid_output.json"
VALID_REPORT_PATH = WORK_DIR / "valid_report.json"
TEST_OUTPUT_PATH = WORK_DIR / "test_predictions.json"

SAFE_EOS_ID = 50256
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_DIR:", WORK_DIR)
print("GENERATED_DATA_DIR:", GENERATED_DATA_DIR)
print("TEST_FILE:", TEST_FILE)


In [ ]:
# 3. Đọc dữ liệu
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None


def load_records(path, need_response=False):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        first = f.read(1)
        f.seek(0)
        records = json.load(f) if first == "[" else [json.loads(line) for line in f if line.strip()]

    out = []
    for i, rec in enumerate(records):
        if "query_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu query_vi")
        if need_response and "response_vi" not in rec:
            raise KeyError(f"Dòng {i} thiếu response_vi")
        item = dict(rec)
        item.setdefault("id", i)
        item.setdefault("type", "unknown")
        out.append(item)
    return out


raw_train = load_records(TRAIN_FILE, need_response=True)
raw_valid = load_records(VALID_FILE, need_response=True) if VALID_FILE.exists() else []
raw_test = load_records(TEST_FILE) if TEST_FILE else []

if MAX_TRAIN_SAMPLES is not None:
    raw_train = raw_train[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES is not None:
    raw_valid = raw_valid[:MAX_VALID_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    raw_test = raw_test[:MAX_TEST_SAMPLES]

print("raw train:", len(raw_train))
print("raw valid:", len(raw_valid))
print("raw test :", len(raw_test))
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2)[:1400])


In [ ]:
# 4. Hàm trích đáp án và tính điểm
ANSWER_ANCHORS = [
    r"Đáp án là\s*[:：]?",
    r"Câu trả lời là\s*[:：]?",
    r"(?:Câu\s+)?Trả lời(?:\s+là)?\s*[:：]?",
    r"Đáp án\s*[:：]?",
    r"The answer is\s*[:：]?",
    r"Answer\s*[:：]?",
    r"####\s*",
]
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")


def clean_answer_tail(text):
    if text is None:
        return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    text = text.strip(" .。;；,，")
    return text or None


def extract_answer_text(text, allow_last_number=False):
    text = str(text or "")
    anchor_re = re.compile(
        r"(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer|####)\s*[:：]?",
        flags=re.IGNORECASE,
    )
    matches = list(anchor_re.finditer(text))
    if matches:
        tail = text[matches[-1].end():]
        return clean_answer_tail(tail)

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    if allow_last_number:
        nums = NUM_RE.findall(text)
        if nums:
            return clean_answer_tail(nums[-1])
    return None


def parse_plain_number(text):
    text = str(text).strip().replace(" ", "")
    if not text:
        return None

    if "/" in text:
        parts = text.split("/")
        if len(parts) == 2:
            a = parse_plain_number(parts[0])
            b = parse_plain_number(parts[1])
            if a is not None and b not in (None, 0):
                return a / b
        return None

    if re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", text):
        text = text.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", text):
        text = text.replace(",", "")
    elif "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    elif "," in text and "." in text:
        text = text.replace(",", "")

    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None


def parse_number(text):
    if text is None:
        return None
    text = str(text).strip()
    if not text:
        return None
    direct = parse_plain_number(text)
    if direct is not None:
        return direct
    m = NUM_RE.search(text)
    return parse_plain_number(m.group(0)) if m else None


def relative_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def score_one(rel_err, extractable=True):
    if not extractable or rel_err is None:
        return 0
    if rel_err <= 0.01:
        return 10
    if rel_err <= 0.10:
        return 5
    if rel_err <= 0.50:
        return 1
    return 0


In [ ]:
# 5. Data processing trước khi train (Bước 1, 2, 3 + dedup nhẹ)
DROP_TRAIN_WITHOUT_FINAL_ANSWER = True
SAVE_PREPROCESSED_TRAIN = True
DEDUP_TRAIN = True   # V2: bật dedup nhẹ để giảm noise
PREPROCESSED_TRAIN_FILE = GENERATED_DATA_DIR / "train_preprocessed.json"
PREPROCESSING_REPORT_FILE = GENERATED_DATA_DIR / "train_preprocessing_report.json"


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def word_count(text):
    return len(re.findall(r"\S+", str(text or "")))


def normalized_hash(text):
    text = normalize_space(text).lower()
    return hashlib.blake2b(text.encode("utf-8"), digest_size=16).hexdigest()


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_records_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def load_jsonl_records(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def dataset_fingerprint(records):
    content = json.dumps(
        [r.get("query_vi", "") + "\n" + r.get("response_vi", "") for r in records],
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.md5(content).hexdigest()


def fix_artifacts(text):
    text = str(text or "")
    text = text.replace(r"\đóng hộp{", r"\boxed{")
    text = text.replace("\u200b", "").replace("\ufeff", "")
    return text.strip()


def strip_asy_blocks(text):
    text = str(text or "")
    text = re.sub(r"\[asy\].*?\[/asy\]", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(
        r"\[asy\].*?(?=(?:Giá trị của|Giá trị là|Câu trả lời|Đáp án|Nếu chúng ta biết|Để giải|$))",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    text = re.sub(r"\[/asy\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def clean_text(text):
    text = str(text or "")
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(Giá trị của biến [^\n?]+\?)\s*\1", r"\1", text)
    if re.search(r"Đáp án là|Câu trả lời là|####|\\boxed", text, flags=re.IGNORECASE):
        text = re.sub(
            r"\n(?:The answer is[:\s]+[\d.,/\\{}a-zA-Z]+\.?\s*)+$",
            "",
            text,
            flags=re.IGNORECASE,
        )
    return text.strip()


def normalize_decimal_format(text):
    text = str(text or "")
    text = re.sub(
        r"(?<![{\\])(\d+)\.(\d{3}),(\d{1,3})(?!\d)",
        lambda m: f"{m.group(1)}{m.group(2)}.{m.group(3)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?0),(\d{1,3})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    text = re.sub(
        r"(?<![a-zA-ZÀ-ỹ{])(-?\d+),(\d{1,2})(?!\d)(?!})",
        lambda m: f"{m.group(1)}.{m.group(2)}",
        text,
    )
    return text


def preprocess_step2(query, response):
    query = strip_asy_blocks(query)
    response = strip_asy_blocks(response)
    query = clean_text(query)
    response = clean_text(response)
    query = normalize_decimal_format(query)
    response = normalize_decimal_format(response)
    return query, response


def extract_final_answer(response):
    text = str(response or "")
    anchor_re = re.compile(
        r"(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer|####)\s*[:：]?",
        flags=re.IGNORECASE,
    )
    matches = list(anchor_re.finditer(text))
    if matches:
        return clean_answer_tail(text[matches[-1].end():])

    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])

    numbers = re.findall(
        r"(?:\\frac\{[^}]+\}\{[^}]+\}|[-+]?\d+(?:[.,]\d+)?(?:\s*\\[a-zA-Z]+\{[^}]*\})*)",
        text,
    )
    if numbers:
        return clean_answer_tail(numbers[-1])
    return None


def normalize_answer(answer):
    answer = clean_answer_tail(answer) or ""
    answer = re.sub(r"\s+", " ", answer).strip()
    answer = re.sub(r"\(([-+]?\d+),([-+]?\d+)\)", r"(\1, \2)", answer)

    if not re.search(r"[\\{^_]", answer):
        if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", answer) and not re.fullmatch(r"[-+]?0,\d{3}", answer):
            answer = answer.replace(",", "")
        elif re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", answer):
            answer = answer.replace(".", "").replace(",", ".")
        else:
            answer = normalize_decimal_format(answer)
        answer = re.sub(r"^([-+]?\d[\d./]*)(?:\s+[a-zA-ZÀ-ỹ%].*)$", r"\1", answer)
    else:
        answer = normalize_decimal_format(answer)

    return answer.strip(" .。;；,，")


def rebuild_response(response, answer):
    cleaned = str(response or "").strip()
    cleaned = re.sub(
        r"\s*(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer)\s*[:：]?\s*[^\n]*\s*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s*####\s*[^\n]*\s*$", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n?\s*\\boxed\s*\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}\s*[.。]?\s*$", "", cleaned)
    cleaned = cleaned.rstrip()
    return (cleaned + f"\nĐáp án là: {answer}").strip()


def preprocess_labeled_record(rec, idx, drop_without_answer):
    query_raw = fix_artifacts(rec.get("query_vi"))
    response_raw = fix_artifacts(rec.get("response_vi"))
    if not query_raw or not response_raw:
        return None, "missing_query_or_response"

    query, response = preprocess_step2(query_raw, response_raw)
    answer = normalize_answer(extract_final_answer(response))
    if not answer and drop_without_answer:
        return None, "extract_failed"

    if answer:
        response = rebuild_response(response, answer)

    item = {
        "id": rec.get("id", idx),
        "query_vi": query,
        "response_vi": response,
        "type": rec.get("type", "unknown"),
        "answer_text": answer or None,
        "answer_num": parse_number(answer) if answer else None,
    }
    return item, None


def process_train(records):
    kept = []
    drop_reasons = []
    failed_samples = []
    asy_stripped = 0

    for i, rec in enumerate(tqdm(records, desc="text preprocessing")):
        raw_joined = f"{rec.get('query_vi', '')}\n{rec.get('response_vi', '')}".lower()
        had_asy = "[asy]" in raw_joined
        item, reason = preprocess_labeled_record(rec, i, DROP_TRAIN_WITHOUT_FINAL_ANSWER)
        if reason:
            drop_reasons.append(reason)
            if reason == "extract_failed" and len(failed_samples) < 50:
                failed_samples.append({
                    "index": i,
                    "type": rec.get("type", "unknown"),
                    "query_vi": normalize_space(rec.get("query_vi"))[:180],
                    "response_tail": str(rec.get("response_vi", ""))[-300:],
                })
            continue
        if had_asy:
            asy_stripped += 1
        kept.append(item)

    return kept, Counter(drop_reasons), {
        "extract_failed_preview": failed_samples,
        "asy_stripped_count": asy_stripped,
    }


def dedup_records(records):
    """V2: dedup exact (query, answer_text). Giữ bản đầu tiên."""
    seen = set()
    out = []
    dup = 0
    for rec in records:
        key = (
            normalize_space(rec.get("query_vi", "")).lower(),
            str(rec.get("answer_text", "")).strip(),
        )
        if key in seen:
            dup += 1
            continue
        seen.add(key)
        out.append(rec)
    return out, dup


def process_eval_or_test(records, has_response):
    out = []
    for i, rec in enumerate(records):
        query_raw = fix_artifacts(rec.get("query_vi"))
        query = normalize_decimal_format(clean_text(strip_asy_blocks(query_raw)))
        item = {
            "id": rec.get("id", i),
            "query_vi": query,
            "type": rec.get("type", "unknown"),
        }
        if has_response:
            response_raw = fix_artifacts(rec.get("response_vi"))
            _, response = preprocess_step2(query_raw, response_raw)
            answer = normalize_answer(extract_final_answer(response))
            if answer:
                response = rebuild_response(response, answer)
            item["response_vi"] = response
            item["answer_text"] = answer or None
            item["answer_num"] = parse_number(answer) if answer else None
        out.append(item)
    return out


train_records, drop_counter, preprocess_logs = process_train(raw_train)
n_before_dedup = len(train_records)
if DEDUP_TRAIN:
    train_records, n_dup = dedup_records(train_records)
    print(f"Dedup: bỏ {n_dup} cặp (query, answer) trùng. Còn {len(train_records)} mẫu.")
else:
    n_dup = 0

valid_records = process_eval_or_test(raw_valid, has_response=True)
test_records = process_eval_or_test(raw_test, has_response=False)

preprocessing_report = {
    "train_before": len(raw_train),
    "train_after_text_preprocessing": n_before_dedup,
    "train_after_dedup": len(train_records),
    "dropped_text_preprocessing": sum(drop_counter.values()),
    "drop_reasons_text_preprocessing": dict(drop_counter),
    "n_duplicates_removed": n_dup,
    "valid": len(valid_records),
    "test": len(test_records),
    "fingerprint_text_preprocessing": dataset_fingerprint(train_records),
    **preprocess_logs,
}

print("train before:", len(raw_train), "| after text preprocessing:", n_before_dedup, "| dropped:", sum(drop_counter.values()))
print("drop reasons:", dict(drop_counter))
print("[asy] stripped in train:", preprocess_logs["asy_stripped_count"])
print("valid:", len(valid_records), "| test:", len(test_records))
print("File train mới sẽ được ghi sau smart truncation:", PREPROCESSED_TRAIN_FILE)
print("\nTarget sau xử lý:")
print(train_records[0]["response_vi"][:800])


In [ ]:
# 6. Kiểm tra dữ liệu sau processing
def feature_df(records, split):
    rows = []
    for i, rec in enumerate(records):
        rows.append({
            "split": split,
            "index": i,
            "type": rec.get("type", "unknown"),
            "query_words": word_count(rec.get("query_vi")),
            "response_words": word_count(rec.get("response_vi", "")),
            "has_answer": rec.get("answer_text") is not None,
            "answer_num": rec.get("answer_num"),
            "ends_with_anchor": str(rec.get("response_vi", "")).rstrip().endswith("Đáp án là: " + str(rec.get("answer_text", ""))),
        })
    return pd.DataFrame(rows)


train_df = feature_df(train_records, "train")
valid_df = feature_df(valid_records, "valid") if valid_records else pd.DataFrame()

print("Phân bố type sau processing:")
display(train_df["type"].value_counts().rename_axis("type").reset_index(name="count"))

print("Độ dài train (words):")
display(train_df[["query_words", "response_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))

print("Độ dài và answer rate theo type:")
by_type = (
    train_df.groupby("type")
    .agg(
        count=("index", "count"),
        query_p95=("query_words", lambda s: s.quantile(0.95)),
        response_p95=("response_words", lambda s: s.quantile(0.95)),
        answer_rate=("has_answer", "mean"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)
display(by_type.round(3))

print("Tỷ lệ có final_answer:", round(float(train_df["has_answer"].mean()), 4))
print("Anchor không nằm cuối response:", int((~train_df["ends_with_anchor"]).sum()) if len(train_df) else 0)


In [ ]:
# 7. Prompt V2, tokenizer, smart truncation, token audit
MAX_LENGTH = 512
TOKEN_AUDIT_SAMPLES = 2000

# V2 PROMPT: instruction + type hint tiếng Việt
TYPE_LABEL_MAP = {
    "GSM_AnsAug":     "Bài toán số học đời sống",
    "GSM_Rephrased":  "Bài toán số học đời sống",
    "MATH_AnsAug":    "Bài toán nâng cao",
    "MATH_Rephrased": "Bài toán nâng cao",
    "MATH_SV":        "Bài toán nâng cao",
    "MATH_FOBAR":     "Bài toán nâng cao",
    "unknown":        "Bài toán",
}

INSTRUCTION = 'Giải bài toán sau từng bước, kết thúc bằng dòng "Đáp án là: <số>".'
PROMPT_TEMPLATE = (
    "{instr}\n\n"
    "[Loại: {type_label}]\n"
    "Bài toán: {q}\n\n"
    "Lời giải:\n"
)


def get_type_label(t):
    return TYPE_LABEL_MAP.get(t, TYPE_LABEL_MAP["unknown"])


def build_prompt(rec):
    return PROMPT_TEMPLATE.format(
        instr=INSTRUCTION,
        type_label=get_type_label(rec.get("type", "unknown")),
        q=str(rec.get("query_vi", "")).strip(),
    )


tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)

# Verify EOS thật của tokenizer; fallback về SAFE_EOS_ID nếu None.
# Lưu ý: checkpoint GPT-2 có thể có embedding nhỏ hơn len(tokenizer),
# nên model sẽ được resize ở cell train/inference trước khi dùng PAD/EOS này.
real_eos = tokenizer.eos_token_id
EOS_ID = int(real_eos) if real_eos is not None else SAFE_EOS_ID
PAD_ID = EOS_ID
tokenizer.pad_token_id = PAD_ID
tokenizer.eos_token_id = EOS_ID
if getattr(tokenizer, "pad_token", None) is None and getattr(tokenizer, "eos_token", None) is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab_size:", getattr(tokenizer, "vocab_size", None), "| len:", len(tokenizer))
print(f"EOS_ID (used): {EOS_ID} | PAD_ID: {PAD_ID} | tokenizer.eos_token_id thật: {real_eos}")


def encode_no_special(text):
    return tokenizer(str(text or ""), add_special_tokens=False)["input_ids"]


def split_response_for_truncation(response, answer):
    response = str(response or "").rstrip()
    m = re.search(r"\nĐáp án là:\s*([^\n]+)\s*$", response, flags=re.IGNORECASE)
    if m:
        return response[:m.start()].rstrip(), "\nĐáp án là: " + m.group(1).strip()
    suffix = "\nĐáp án là: " + str(answer or extract_answer_text(response, allow_last_number=True) or "").strip()
    body = re.sub(r"\n?Đáp án là:\s*[^\n]+\s*$", "", response, flags=re.IGNORECASE).rstrip()
    return body, suffix


def measure_record_tokens(rec):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(rec.get("response_vi", "")) + [EOS_ID]
    return len(prompt_ids), len(response_ids), len(prompt_ids) + len(response_ids)


def smart_truncate_record(rec, max_length):
    prompt_ids = encode_no_special(build_prompt(rec))
    response_ids = encode_no_special(rec.get("response_vi", "")) + [EOS_ID]
    original_length = len(prompt_ids) + len(response_ids)

    item = dict(rec)
    item["original_length"] = original_length
    item["was_truncated"] = False

    if original_length <= max_length:
        item["prompt_tokens"] = len(prompt_ids)
        item["response_tokens"] = len(response_ids)
        item["total_tokens"] = original_length
        return item, None

    body, suffix = split_response_for_truncation(rec.get("response_vi", ""), rec.get("answer_text"))
    suffix_ids = encode_no_special(suffix) + [EOS_ID]
    middle_budget = max_length - len(prompt_ids) - len(suffix_ids)
    if middle_budget <= 0:
        return None, "too_long_prompt_or_answer"

    body_ids = encode_no_special(body)
    while True:
        kept_body_ids = body_ids[-middle_budget:] if len(body_ids) > middle_budget else body_ids
        body_text = tokenizer.decode(kept_body_ids).strip()
        new_response = (body_text.rstrip() + suffix) if body_text else suffix.lstrip()
        new_response_ids = encode_no_special(new_response) + [EOS_ID]
        new_total = len(prompt_ids) + len(new_response_ids)
        if new_total <= max_length:
            item["response_vi"] = new_response
            item["was_truncated"] = True
            item["prompt_tokens"] = len(prompt_ids)
            item["response_tokens"] = len(new_response_ids)
            item["total_tokens"] = new_total
            return item, None
        overflow = new_total - max_length
        middle_budget -= max(1, overflow)
        if middle_budget <= 0:
            return None, "too_long_after_truncation"


def apply_token_length_policy(records, max_length):
    kept = []
    counter = Counter()
    examples = []
    for rec in tqdm(records, desc="smart truncation"):
        item, reason = smart_truncate_record(rec, max_length)
        if reason:
            counter[reason] += 1
            if len(examples) < 20:
                examples.append({
                    "id": rec.get("id"),
                    "type": rec.get("type"),
                    "reason": reason,
                    "query_vi": rec.get("query_vi", "")[:180],
                })
            continue
        if item.get("was_truncated"):
            counter["smart_truncated"] += 1
        kept.append(item)
    return kept, counter, examples


train_records, token_policy_counter, token_policy_examples = apply_token_length_policy(train_records, MAX_LENGTH)
preprocessing_report.update({
    "max_length": MAX_LENGTH,
    "train_after_token_policy": len(train_records),
    "dropped_token_policy": int(token_policy_counter.get("too_long_prompt_or_answer", 0) + token_policy_counter.get("too_long_after_truncation", 0)),
    "smart_truncated": int(token_policy_counter.get("smart_truncated", 0)),
    "token_policy_counter": dict(token_policy_counter),
    "token_policy_drop_preview": token_policy_examples,
    "fingerprint_final": dataset_fingerprint(train_records),
    "prompt_template_v2": PROMPT_TEMPLATE,
    "instruction": INSTRUCTION,
    "type_label_map": TYPE_LABEL_MAP,
})

if SAVE_PREPROCESSED_TRAIN:
    save_records_jsonl(train_records, PREPROCESSED_TRAIN_FILE)
    save_json(preprocessing_report, PREPROCESSING_REPORT_FILE)
    print("Wrote:", PREPROCESSED_TRAIN_FILE)
    print("Wrote:", PREPROCESSING_REPORT_FILE)

    train_records = load_jsonl_records(PREPROCESSED_TRAIN_FILE)
    TRAIN_SOURCE = f"preprocessed_file:{PREPROCESSED_TRAIN_FILE}"
else:
    TRAIN_SOURCE = "preprocessed_in_memory"

print("TRAIN_SOURCE:", TRAIN_SOURCE)
print("Train records used by Trainer:", len(train_records))
assert train_records, "Không có mẫu train sau preprocessing"
assert max(r.get("total_tokens", 0) for r in train_records) <= MAX_LENGTH, "Còn mẫu vượt MAX_LENGTH"

sample = train_records if len(train_records) <= TOKEN_AUDIT_SAMPLES else random.sample(train_records, TOKEN_AUDIT_SAMPLES)
token_rows = []
for rec in tqdm(sample, desc="token audit"):
    p_tokens, r_tokens, total_tokens = measure_record_tokens(rec)
    token_rows.append({
        "type": rec.get("type"),
        "prompt_tokens": p_tokens,
        "response_tokens": r_tokens,
        "total_tokens": total_tokens,
        "will_truncate": total_tokens > MAX_LENGTH,
        "was_truncated": bool(rec.get("was_truncated")),
    })

token_df = pd.DataFrame(token_rows)
display(token_df[["prompt_tokens", "response_tokens", "total_tokens"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
print("Tỷ lệ còn vượt MAX_LENGTH:", round(float(token_df["will_truncate"].mean()), 4))
print("Số mẫu smart truncated:", int(token_policy_counter.get("smart_truncated", 0)))
print("Số mẫu drop vì quá dài:", preprocessing_report["dropped_token_policy"])

print("\n--- Sample prompt V2 ---")
print(build_prompt(train_records[0]))
print("--- end ---")


In [ ]:
# 8. Dataset cho supervised fine-tuning (loss mask trên prompt)
def clamp_ids(ids, vocab_size):
    return [min(max(int(x), 0), vocab_size - 1) for x in ids]


def fit_prompt_response(prompt_ids, response_ids, max_length):
    if len(prompt_ids) + len(response_ids) <= max_length:
        return prompt_ids, response_ids
    room = max_length - len(prompt_ids)
    if room <= 0:
        prompt_ids = prompt_ids[: max_length - 1]
        room = max_length - len(prompt_ids)
    response_ids = response_ids[-room:] if room > 0 else []
    return prompt_ids, response_ids


class MathDataset(Dataset):
    def __init__(self, records, tokenizer, vocab_size, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.vocab_size = vocab_size
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = self.tokenizer(build_prompt(rec), add_special_tokens=False)["input_ids"]
        response_ids = self.tokenizer(rec["response_vi"], add_special_tokens=False)["input_ids"] + [EOS_ID]
        prompt_ids, response_ids = fit_prompt_response(prompt_ids, response_ids, self.max_length)

        input_ids = clamp_ids(prompt_ids + response_ids, self.vocab_size)
        labels = [-100] * len(prompt_ids) + clamp_ids(response_ids, self.vocab_size)
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        max_len = int(math.ceil(max_len / 8) * 8)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [ ]:
# 9. Tham số train V2 (2 epochs, batch eff 32, LR 5e-5)
RUN_TRAIN = True
EPOCHS = 2                       # V2: 1 -> 2
PER_DEVICE_BATCH_SIZE = 8        # V2: 4 -> 8 (giảm về 4 nếu OOM)
GRAD_ACCUM = 4                   # effective batch = 8 * 4 = 32
LEARNING_RATE = 5e-5             # V2: 3e-5 -> 5e-5
WARMUP_RATIO = 0.03              # V2: 0.05 -> 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 50
TRAINER_EVAL_SAMPLES = 500       # V2: 1000 -> 500 (eval loss đỡ tốn thời gian)


def ensure_model_token_embeddings(model, tok, pad_id, eos_id):
    """Resize embedding nếu tokenizer có token id ngoài vocab của checkpoint."""
    token_count = len(tok)
    embed_count = model.get_input_embeddings().num_embeddings
    if token_count > embed_count:
        print(f"Resize token embeddings: {embed_count} -> {token_count}")
        model.resize_token_embeddings(token_count)
        embed_count = model.get_input_embeddings().num_embeddings

    special_ids = [int(x) for x in [pad_id, eos_id] if x is not None]
    max_special_id = max(special_ids) if special_ids else -1
    if max_special_id >= embed_count:
        raise ValueError(
            f"PAD/EOS id ngoài embedding vocab: max_special_id={max_special_id}, "
            f"embedding_size={embed_count}. Hãy kiểm tra tokenizer/model hoặc resize embedding."
        )

    model.config.pad_token_id = int(pad_id) if pad_id is not None else None
    model.config.eos_token_id = int(eos_id) if eos_id is not None else None
    return model


def audit_dataset_batch(dataset, data_collator, vocab_size, name):
    if dataset is None or len(dataset) == 0:
        return
    sample_size = min(8, len(dataset))
    batch = [dataset[i] for i in range(sample_size)]
    tensors = data_collator(batch)
    input_ids = tensors["input_ids"]
    labels = tensors["labels"]
    input_min = int(input_ids.min().item())
    input_max = int(input_ids.max().item())
    valid_labels = labels[labels != -100]
    label_min = int(valid_labels.min().item()) if valid_labels.numel() else -100
    label_max = int(valid_labels.max().item()) if valid_labels.numel() else -100
    if input_min < 0 or input_max >= vocab_size or label_max >= vocab_size:
        raise ValueError(
            f"{name} batch có token id ngoài vocab: "
            f"input_range=[{input_min}, {input_max}], "
            f"label_range=[{label_min}, {label_max}], vocab_size={vocab_size}"
        )
    print(
        f"{name} token audit OK | input_range=[{input_min}, {input_max}] "
        f"| label_range=[{label_min}, {label_max}] | vocab_size={vocab_size}"
    )


tmp_model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
tmp_model = ensure_model_token_embeddings(tmp_model, tokenizer, PAD_ID, EOS_ID)
MODEL_VOCAB_SIZE = tmp_model.get_input_embeddings().num_embeddings
del tmp_model
gc.collect()
torch.cuda.empty_cache()

train_ds = MathDataset(train_records, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH)
eval_records_for_trainer = valid_records[:TRAINER_EVAL_SAMPLES]
# Eval dataset cần có response_vi cho mỗi mẫu (sau preprocess đã có).
eval_records_filtered = [r for r in eval_records_for_trainer if r.get("response_vi")]
eval_ds = MathDataset(eval_records_filtered, tokenizer, MODEL_VOCAB_SIZE, MAX_LENGTH) if eval_records_filtered else None
collator = PadCollator(pad_id=PAD_ID)
audit_dataset_batch(train_ds, collator, MODEL_VOCAB_SIZE, "train")
audit_dataset_batch(eval_ds, collator, MODEL_VOCAB_SIZE, "eval")

effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count() if CUDA_OK else 0)
steps_per_epoch = math.ceil(len(train_ds) / effective_batch)
total_train_steps = steps_per_epoch * EPOCHS
WARMUP_STEPS = max(1, int(total_train_steps * WARMUP_RATIO)) if total_train_steps > 0 else 0
print("train source:", globals().get("TRAIN_SOURCE", "train_records"))
print("preprocessed train file:", PREPROCESSED_TRAIN_FILE)
print("train samples:", len(train_ds), "| eval samples:", len(eval_ds) if eval_ds else 0)
print("effective batch:", effective_batch)
print("steps/epoch:", steps_per_epoch)
print(f"Total steps for {EPOCHS} epochs:", total_train_steps)
print("warmup steps:", WARMUP_STEPS)


def make_training_args():
    use_bf16 = bool(CUDA_OK and torch.cuda.is_bf16_supported())
    use_fp16 = bool(CUDA_OK and not use_bf16)
    kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=LOGGING_STEPS,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=2 if IS_KAGGLE else 0,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch" if eval_ds else "no"
    else:
        kwargs["evaluation_strategy"] = "epoch" if eval_ds else "no"
    if "bf16" in sig.parameters:
        kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters:
        kwargs["fp16"] = use_fp16
    # V2: dùng adamw_torch_fused trên CUDA hiện đại
    if "optim" in sig.parameters and CUDA_OK:
        kwargs["optim"] = "adamw_torch_fused"
    if not CUDA_OK:
        if "use_cpu" in sig.parameters:
            kwargs["use_cpu"] = True
        elif "no_cuda" in sig.parameters:
            kwargs["no_cuda"] = True
    return TrainingArguments(**kwargs)


In [ ]:
# 10. Train và lưu checkpoint
if RUN_TRAIN:
    if not CUDA_OK:
        raise RuntimeError(
            "Không có GPU CUDA dùng được cho full training. Hãy đổi Accelerator sang T4/V100/A100."
        )
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = ensure_model_token_embeddings(model, tokenizer, PAD_ID, EOS_ID)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        args=make_training_args(),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
    )

    start = time.time()
    train_output = trainer.train()
    print("Train minutes:", round((time.time() - start) / 60, 2))
    print(train_output)

    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skip train. Inference sẽ dùng checkpoint nếu có, nếu không dùng base model.")


In [ ]:
# 11. Hàm sinh lời giải V2 — batch + beam search + custom StoppingCriteria
MAX_NEW_TOKENS = 320              # V2: 192 -> 320
NUM_BEAMS = 4                     # V2: 1 -> 4 (beam search)
DO_SAMPLE = False
REPETITION_PENALTY = 1.0          # V2: 1.05 -> 1.0 (không phạt)
NO_REPEAT_NGRAM_SIZE = 0          # V2: 4 -> 0 (TẮT)
INFER_BATCH_SIZE = 8              # V2: batch inference
EARLY_STOPPING = True


class AnswerStoppingCriteria(StoppingCriteria):
    """Dừng generation khi mọi sequence trong batch đã có dòng 'Đáp án là: <số>\n' hoặc EOS.

    Chỉ decode 80 token cuối cho hiệu năng.
    """
    def __init__(self, tokenizer, prompt_lens, eos_id, check_every=8):
        super().__init__()
        self.tokenizer = tokenizer
        self.prompt_lens = prompt_lens  # list[int], len(prompt_lens) = batch_size * num_beams (or batch_size)
        self.eos_id = eos_id
        self.pattern = re.compile(r"Đáp án là\s*[:：]?\s*[-+]?\d", re.IGNORECASE)
        self.check_every = check_every
        self._step = 0

    def __call__(self, input_ids, scores, **kwargs):
        self._step += 1
        if self._step % self.check_every != 0:
            return False
        # input_ids shape: (batch * num_beams, seq_len) for beam search
        bsz = input_ids.shape[0]
        done = []
        for i in range(bsz):
            seq = input_ids[i]
            # Lấy phần generated (sau prompt). Beam search có thể có effective length khác.
            # Lấy 80 token cuối là đủ để bắt anchor.
            tail = seq[-80:].tolist()
            text = self.tokenizer.decode(tail, skip_special_tokens=True)
            # Đã có anchor + ít nhất 1 chữ số tiếp theo
            if self.pattern.search(text):
                # Đảm bảo có ký tự kết thúc câu sau số (\n hoặc end-of-text), cho phép dừng
                done.append(True)
            else:
                done.append(False)
        return all(done)


def save_json(obj, path):
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def postprocess_output(text):
    """Cắt phần thừa sau khi model emit câu trả lời. Giữ phần đầu tiên có anchor."""
    text = str(text).strip()
    # Cắt khi gặp marker bắt đầu example mới
    for marker in ["\nCâu hỏi:", "\nQuestion:", "\nBài toán:", "\n[Loại:", "\n###", "\nGiải bài toán"]:
        pos = text.find(marker)
        if pos >= 0:
            text = text[:pos].strip()
    # Cắt sau dòng "Đáp án là: <số>" đầu tiên (giữ lại dòng đó)
    m = re.search(r"Đáp án là\s*[:：]?\s*[^\n]+", text, flags=re.IGNORECASE)
    if m:
        text = text[: m.end()].strip()
    return text


def ensure_anchor(text):
    """Nếu output không có anchor nhưng có số cuối, tự gắn 'Đáp án là: <số>'."""
    if re.search(r"Đáp án là\s*[:：]?", text, flags=re.IGNORECASE):
        return text
    last_num = extract_answer_text(text, allow_last_number=True)
    if last_num:
        return text.rstrip() + f"\nĐáp án là: {last_num}"
    return text


def generate_predictions(model_dir, records, output_path, name):
    device = "cuda" if CUDA_OK else "cpu"
    print("Load for generation:", model_dir, "| device:", device)

    gen_tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
    gen_tokenizer.pad_token_id = PAD_ID
    gen_tokenizer.eos_token_id = EOS_ID
    # Left-padding cần thiết cho batch inference với causal LM
    gen_tokenizer.padding_side = "left"
    if getattr(gen_tokenizer, "pad_token", None) is None and getattr(gen_tokenizer, "eos_token", None) is not None:
        gen_tokenizer.pad_token = gen_tokenizer.eos_token

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(str(model_dir), torch_dtype=dtype, local_files_only=True)
    model = ensure_model_token_embeddings(model, gen_tokenizer, PAD_ID, EOS_ID).to(device)
    model.config.use_cache = True
    model.eval()

    vocab_size = model.get_input_embeddings().num_embeddings
    outputs = []
    start_all = time.time()

    # Sort records by query length để batch tương đối đồng đều (giảm padding waste)
    order = sorted(range(len(records)), key=lambda i: len(records[i].get("query_vi", "")))
    sorted_records = [records[i] for i in order]

    with torch.inference_mode():
        for batch_start in tqdm(range(0, len(sorted_records), INFER_BATCH_SIZE), desc=name):
            batch = sorted_records[batch_start: batch_start + INFER_BATCH_SIZE]
            prompts = [build_prompt(r) for r in batch]
            enc = gen_tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(device)
            input_ids = enc["input_ids"].clamp(min=0, max=vocab_size - 1)
            attention_mask = enc["attention_mask"]
            prompt_lens = attention_mask.sum(dim=1).tolist()

            stopping = StoppingCriteriaList([
                AnswerStoppingCriteria(gen_tokenizer, prompt_lens, EOS_ID, check_every=8)
            ])

            gen = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                num_beams=NUM_BEAMS,
                early_stopping=EARLY_STOPPING,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID,
                stopping_criteria=stopping,
            )

            # Với left padding, prompt nằm bên trái, generated = từ input_ids.shape[1] trở đi
            prompt_len_tensor = input_ids.shape[1]
            for i, rec in enumerate(batch):
                new_tokens = gen[i, prompt_len_tensor:]
                text = gen_tokenizer.decode(new_tokens, skip_special_tokens=True)
                text = postprocess_output(text)
                text = ensure_anchor(text)
                outputs.append({
                    "id": rec.get("id"),
                    "query_vi": rec["query_vi"],
                    "type": rec.get("type", "unknown"),
                    "model_output": text,
                    "_sort_index": batch_start + i,
                })

    # Sắp xếp lại theo thứ tự gốc (map qua order)
    inv_order = {sorted_idx: orig_idx for orig_idx, sorted_idx in enumerate(order)}
    outputs_final = [None] * len(outputs)
    for o in outputs:
        sorted_pos = o.pop("_sort_index")
        original_pos = order[sorted_pos]
        outputs_final[original_pos] = o
    outputs_final = [o for o in outputs_final if o is not None]

    save_json(outputs_final, output_path)
    print("Wrote:", output_path)
    print("Minutes:", round((time.time() - start_all) / 60, 2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return outputs_final


In [ ]:
# 12. Sinh output validation
RUN_VALIDATION = True
FORCE_REGENERATE_VALID = False


def checkpoint_ready(path):
    path = Path(path)
    has_config = (path / "config.json").exists()
    has_tokenizer = (path / "tokenizer.json").exists() or (path / "vocab.json").exists()
    has_weights = (path / "model.safetensors").exists() or (path / "pytorch_model.bin").exists()
    return path.exists() and has_config and has_tokenizer and has_weights


MODEL_FOR_INFERENCE = OUTPUT_DIR if checkpoint_ready(OUTPUT_DIR) else MODEL_DIR

print("RUN_VALIDATION:", RUN_VALIDATION)
print("VALID_FILE:", VALID_FILE, "| exists:", VALID_FILE.exists())
print("raw_valid:", len(raw_valid) if "raw_valid" in globals() else "MISSING")
print("valid_records:", len(valid_records) if "valid_records" in globals() else "MISSING")
print("OUTPUT_DIR:", OUTPUT_DIR, "| checkpoint_ready:", checkpoint_ready(OUTPUT_DIR))
print("MODEL_FOR_INFERENCE:", MODEL_FOR_INFERENCE)
print("VALID_OUTPUT_PATH:", VALID_OUTPUT_PATH, "| exists:", VALID_OUTPUT_PATH.exists())

if not valid_records:
    valid_outputs = []
    print("Skip validation generation: valid_records đang rỗng. Kiểm tra lại DATA_DIR/VALID_FILE và cell đọc dữ liệu.")
elif VALID_OUTPUT_PATH.exists() and not FORCE_REGENERATE_VALID:
    with Path(VALID_OUTPUT_PATH).open("r", encoding="utf-8") as f:
        valid_outputs = json.load(f)
    print("Loaded existing validation output:", VALID_OUTPUT_PATH, "| n=", len(valid_outputs))
elif RUN_VALIDATION:
    valid_outputs = generate_predictions(MODEL_FOR_INFERENCE, valid_records, VALID_OUTPUT_PATH, name="validation")
else:
    valid_outputs = []
    print("Skip validation generation: RUN_VALIDATION=False")

if valid_outputs:
    print("\nOutput mẫu:")
    print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:1200])
else:
    print("WARNING: valid_outputs vẫn rỗng, nên cell đánh giá sẽ không có gì để chấm.")



In [ ]:
# 13. Đánh giá validation + lưu valid_report.json
CASES_TO_SHOW = 8


def align_by_id(preds, golds):
    if all("id" in x for x in preds) and all("id" in x for x in golds):
        pred_map = {str(x["id"]): x for x in preds}
        return [(pred_map[str(g["id"])], g) for g in golds if str(g["id"]) in pred_map]
    return list(zip(preds, golds))


def evaluate_predictions(preds, golds):
    rows = []
    for row_index, (pred, gold) in enumerate(align_by_id(preds, golds)):
        gold_answer = extract_answer_text(gold.get("response_vi"), allow_last_number=True)
        # V2 FIX: allow_last_number=True cho cả pred (fair) — đã có ensure_anchor() upstream nhưng vẫn cần fallback
        pred_answer = extract_answer_text(pred.get("model_output"), allow_last_number=True)
        gold_num = parse_number(gold_answer)
        pred_num = parse_number(pred_answer)
        rel_err = relative_error(pred_num, gold_num)
        score = score_one(rel_err, pred_answer is not None)
        rows.append({
            "row_index": row_index,
            "id": gold.get("id"),
            "type": gold.get("type"),
            "query_vi": gold.get("query_vi"),
            "model_output": pred.get("model_output"),
            "gold_answer": gold_answer,
            "pred_answer": pred_answer,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": rel_err,
            "extractable": pred_answer is not None,
            "score": score,
        })
    return rows


def score_summary(rows):
    n = len(rows)
    raw = sum(r["score"] for r in rows)
    return {
        "n": n,
        "raw_score": raw,
        "max_raw_score": 10 * n,
        "score_10": raw / n if n else 0,
        "extractable_rate": sum(r["extractable"] for r in rows) / n if n else 0,
        "buckets": {str(s): sum(r["score"] == s for r in rows) for s in [10, 5, 1, 0]},
    }


def show_cases(title, rows):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if not rows:
        print("Không có case")
        return
    cols = ["row_index", "id", "type", "score", "rel_error", "gold_answer", "pred_answer", "query_vi", "model_output"]
    display(pd.DataFrame(rows[:CASES_TO_SHOW])[cols])


if "valid_outputs" not in globals():
    valid_outputs = []

if not valid_outputs and Path(VALID_OUTPUT_PATH).exists():
    with Path(VALID_OUTPUT_PATH).open("r", encoding="utf-8") as f:
        valid_outputs = json.load(f)
    print("Loaded validation output before scoring:", VALID_OUTPUT_PATH, "| n=", len(valid_outputs))

if not valid_records:
    eval_rows = []
    summary = None
    print("Không có valid_records để đánh giá. VALID_FILE:", VALID_FILE, "| exists:", VALID_FILE.exists())
elif valid_outputs:
    eval_rows = evaluate_predictions(valid_outputs, valid_records)
    summary = score_summary(eval_rows)
    print("Validation summary:")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    by_type = pd.DataFrame(eval_rows).groupby("type").apply(lambda x: pd.Series(score_summary(x.to_dict("records")))).reset_index()
    display(by_type[["type", "n", "score_10", "extractable_rate", "raw_score", "max_raw_score", "buckets"]])

    show_cases("Một vài case đúng (score=10)", [r for r in eval_rows if r["score"] == 10])
    show_cases("Một vài case sai (score=0)", [r for r in eval_rows if r["score"] == 0 and r["extractable"]])
    show_cases("Một vài case không tách được đáp án", [r for r in eval_rows if not r["extractable"]])

    # Lưu valid_report.json
    report = {
        "summary": summary,
        "by_type": by_type.to_dict("records"),
        "config": {
            "epochs": EPOCHS,
            "lr": LEARNING_RATE,
            "effective_batch": effective_batch,
            "max_length": MAX_LENGTH,
            "max_new_tokens": MAX_NEW_TOKENS,
            "num_beams": NUM_BEAMS,
            "prompt_template": PROMPT_TEMPLATE,
            "instruction": INSTRUCTION,
        },
        "wrong_cases_preview": [r for r in eval_rows if r["score"] == 0][:30],
    }
    save_json(report, VALID_REPORT_PATH)
    print(f"\nWrote: {VALID_REPORT_PATH}")
else:
    eval_rows = []
    summary = None
    print("Không có validation output để đánh giá.")
    print("Hãy chạy cell 'Sinh output validation' ở ngay phía trên, hoặc kiểm tra lỗi generation.")
    print("VALID_OUTPUT_PATH:", VALID_OUTPUT_PATH, "| exists:", Path(VALID_OUTPUT_PATH).exists())
    print("valid_records:", len(valid_records))



In [ ]:
# 14. Sinh test_predictions.json cho Phase 2
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE and test_records:
    test_outputs = generate_predictions(MODEL_FOR_INFERENCE, test_records, TEST_OUTPUT_PATH, name="test")
    # Loại bỏ field _sort_index nếu còn (defensive)
    test_outputs_clean = [
        {k: v for k, v in o.items() if not k.startswith("_")}
        for o in test_outputs
    ]
    save_json(test_outputs_clean, TEST_OUTPUT_PATH)
    print("\nTest output mẫu:")
    print(json.dumps(test_outputs_clean[:2], ensure_ascii=False, indent=2)[:1200])
else:
    print("Không có test.json, bỏ qua bước test inference")


In [ ]:
# 15. Kiểm tra file đầu ra
for p in [OUTPUT_DIR, VALID_OUTPUT_PATH, VALID_REPORT_PATH, TEST_OUTPUT_PATH]:
    p = Path(p)
    if p.exists():
        size = p.stat().st_size if p.is_file() else "<dir>"
        print(p, "|", size)

print("\nDone.")
